# Topic 5: Production Best Practices

> **Phase 7 → Week 15 → Topic 5**

---

## What Makes a Pipeline "Production-Ready"?

```
Works on my machine   → Correct output on happy path
Production-ready      → Correct output + handles failure + observable + testable
                        + idempotent + documented + deployable without manual steps
```

---

## Interview Cheat Sheet

**Q: What makes a Spark pipeline production-ready?**
> (1) **Idempotent**: re-running produces the same result (dynamic partition overwrite or Delta MERGE). (2) **Observable**: structured logging, metrics to CloudWatch, Spark event logs for History Server. (3) **Testable**: pure transformation functions, pytest with session-scoped SparkSession, >80% coverage. (4) **Configurable**: env-specific config (dev/staging/prod) via args or SSM, no hardcoded paths. (5) **Monitored**: DQ checks after each layer, freshness alerting, runbooks for every alert. (6) **CI/CD**: lint + test on every PR, deploy on merge to main.

**Q: How do you manage configuration across environments (dev/staging/prod)?**
> Never hardcode paths or environment-specific values. Options: (1) CLI arguments (`--env prod --date 2024-01-15`) — simple, auditable. (2) Config files per env (`config/dev.yaml`, `config/prod.yaml`) — checked in, no secrets. (3) AWS SSM Parameter Store — centralized, versioned, IAM-controlled, supports secrets. (4) Airflow Variables — per-environment Airflow deployment with different variable values. Avoid: environment variables set manually, hardcoded `if env == 'prod'` branches.

**Q: How do you handle secrets (database passwords, API keys) in a Spark pipeline?**
> Never hardcode secrets. Never put them in `spark-submit` args (visible in process list). Options: (1) AWS Secrets Manager — call `boto3.client('secretsmanager').get_secret_value()` at runtime. (2) AWS SSM Parameter Store with `SecureString` — IAM controls who can decrypt. (3) Databricks secret scope — `dbutils.secrets.get(scope='prod', key='db_password')`. (4) Kubernetes Secrets (if running on k8s). The Spark job fetches the secret at runtime, it never touches disk or logs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import logging, argparse, sys

spark = SparkSession.builder \
    .appName("Week15 - Production Best Practices") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Production Best Practices — ready")

## Part 1: Structured Logging

In [ ]:
import json
from datetime import datetime, timezone

class StructuredLogger:
    """Emit JSON log lines for easy parsing in CloudWatch/Splunk/Datadog."""

    def __init__(self, pipeline: str, layer: str, run_id: str):
        self.pipeline = pipeline
        self.layer    = layer
        self.run_id   = run_id
        self._logger  = logging.getLogger(f"{pipeline}.{layer}")
        if not self._logger.handlers:
            h = logging.StreamHandler()
            h.setFormatter(logging.Formatter("%(message)s"))
            self._logger.addHandler(h)
            self._logger.setLevel(logging.INFO)

    def _emit(self, level: str, msg: str, **extra):
        record = {
            "ts":       datetime.now(timezone.utc).isoformat(),
            "level":    level,
            "pipeline": self.pipeline,
            "layer":    self.layer,
            "run_id":   self.run_id,
            "msg":      msg,
            **extra
        }
        getattr(self._logger, level.lower())(json.dumps(record))

    def info(self, msg, **kw):    self._emit("INFO",  msg, **kw)
    def warn(self, msg, **kw):    self._emit("WARN",  msg, **kw)
    def error(self, msg, **kw):   self._emit("ERROR", msg, **kw)

    def job_start(self, date: str):
        self.info("Job started", date=date)

    def job_complete(self, date: str, rows_written: int, duration_s: float):
        self.info("Job complete", date=date, rows_written=rows_written,
                  duration_s=round(duration_s, 2))

    def row_count(self, layer: str, count: int):
        self.info("Row count", layer=layer, count=count)


# Demo
import time
log = StructuredLogger(pipeline="orders_etl", layer="silver", run_id="run_20240115_001")
log.job_start(date="2024-01-15")
log.row_count(layer="bronze", count=50000)
log.row_count(layer="silver", count=49800)
log.warn("null_rate_elevated", column="region", null_rate=0.03, threshold=0.01)
log.job_complete(date="2024-01-15", rows_written=49800, duration_s=45.3)

## Part 2: Configuration Management

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class PipelineConfig:
    env:           str
    date:          str
    bronze_path:   str
    silver_path:   str
    gold_path:     str
    dq_path:       str
    shuffle_parts: int
    max_null_rate: float

CONFIGS = {
    "dev": {
        "bronze_path":    "/tmp/dev/bronze",
        "silver_path":    "/tmp/dev/silver",
        "gold_path":      "/tmp/dev/gold",
        "dq_path":        "/tmp/dev/dq",
        "shuffle_parts":  4,
        "max_null_rate":  0.05,  # more lenient in dev
    },
    "staging": {
        "bronze_path":    "s3://my-staging-bucket/bronze",
        "silver_path":    "s3://my-staging-bucket/silver",
        "gold_path":      "s3://my-staging-bucket/gold",
        "dq_path":        "s3://my-staging-bucket/dq",
        "shuffle_parts":  100,
        "max_null_rate":  0.01,
    },
    "prod": {
        "bronze_path":    "s3://my-prod-bucket/bronze",
        "silver_path":    "s3://my-prod-bucket/silver",
        "gold_path":      "s3://my-prod-bucket/gold",
        "dq_path":        "s3://my-prod-bucket/dq",
        "shuffle_parts":  200,
        "max_null_rate":  0.01,
    },
}

def load_config(env: str, date: str) -> PipelineConfig:
    if env not in CONFIGS:
        raise ValueError(f"Unknown env: {env}. Valid: {list(CONFIGS)}")
    cfg = CONFIGS[env].copy()
    return PipelineConfig(env=env, date=date, **cfg)

# Simulate CLI argument parsing
config = load_config(env="dev", date="2024-01-15")
print(f"Config for {config.env}/{config.date}:")
for k, v in config.__dict__.items():
    print(f"  {k:20s}: {v}")

## Part 3: Secret Management

In [ ]:
print("""
Secret Management — AWS Secrets Manager:
══════════════════════════════════════════════════════════════

NEVER do:
  DB_PASSWORD = 'my-secret-password'  # hardcoded
  os.environ['DB_PASSWORD']           # set in shell, leaks to logs
  spark-submit --conf spark.password=abc  # visible in process list

DO — AWS Secrets Manager:
  import boto3, json

  def get_secret(secret_name: str, region='us-east-1') -> dict:
      client = boto3.client('secretsmanager', region_name=region)
      response = client.get_secret_value(SecretId=secret_name)
      return json.loads(response['SecretString'])

  # In your Spark job (runs once per driver, not per task):
  creds = get_secret('prod/orders_etl/db_creds')
  jdbc_url = f"jdbc:postgresql://db:5432/orders?user={creds['username']}&password={creds['password']}"

  # EMR IAM role must have secretsmanager:GetSecretValue permission

DO — AWS SSM Parameter Store (for non-secret config):
  ssm = boto3.client('ssm')

  def get_param(name: str) -> str:
      return ssm.get_parameter(Name=name, WithDecryption=True)['Parameter']['Value']

  code_version  = get_param('/orders_etl/code_version')
  slack_webhook = get_param('/orders_etl/slack_webhook')  # SecureString

DO — Databricks Secret Scope:
  password = dbutils.secrets.get(scope='prod-secrets', key='db_password')
  # secret is redacted from notebook output — never printed even if you try

Secret rotation:
  AWS Secrets Manager can auto-rotate secrets (Lambda-based rotation)
  Your Spark job always fetches at runtime → gets the rotated value automatically
  No redeployment needed after rotation
══════════════════════════════════════════════════════════════
""")

## Part 4: Production Spark Job Template (Complete)

In [ ]:
print("""
Complete Production Spark Job Template:
══════════════════════════════════════════════════════════════

#!/usr/bin/env python3
"""Silver orders transformation — production-ready."""
import argparse, sys, time, logging, json
from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

# ── Logging ────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='{"ts": "%(asctime)s", "level": "%(levelname)s", "msg": %(message)s}',
)
log = logging.getLogger('silver_orders')

# ── Config ─────────────────────────────────────────────────────
def parse_args():
    p = argparse.ArgumentParser(description='Silver orders transform')
    p.add_argument('--date',         required=True, help='YYYY-MM-DD')
    p.add_argument('--env',          default='prod', choices=['dev','staging','prod'])
    p.add_argument('--input-bucket', required=True)
    p.add_argument('--output-bucket',required=True)
    return p.parse_args()

# ── Spark ──────────────────────────────────────────────────────
def build_spark(env: str, shuffle_parts: int) -> SparkSession:
    return (
        SparkSession.builder
        .appName(f'silver_orders_{env}')
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.shuffle.partitions', str(shuffle_parts))
        .config('spark.hadoop.fs.s3a.committer.name', 'magic')
        .getOrCreate()
    )

# ── Transformations (pure functions — testable) ─────────────────
def validate(df: DataFrame) -> DataFrame:
    return df.filter(
        F.col('customer_id').isNotNull() & (F.col('amount') > 0)
    )

def deduplicate(df: DataFrame) -> DataFrame:
    return df.dropDuplicates(['order_id'])

def enrich(df: DataFrame) -> DataFrame:
    return df.withColumn('tier',
        F.when(F.col('amount') >= 500, 'gold')
         .when(F.col('amount') >= 200, 'silver')
         .otherwise('bronze')
    )

# ── DQ ─────────────────────────────────────────────────────────
def run_dq(df: DataFrame, max_null_rate: float) -> None:
    total     = df.count()
    null_cust = df.filter(F.col('customer_id').isNull()).count()
    null_rate = null_cust / total if total else 0
    if null_rate > max_null_rate:
        raise ValueError(f'DQ failed: customer_id null_rate={null_rate:.2%} > {max_null_rate:.2%}')
    log.info(json.dumps({'dq': 'passed', 'total': total, 'null_rate': null_rate}))

# ── Main ────────────────────────────────────────────────────────
def main():
    args   = parse_args()
    cfg    = CONFIGS[args.env]  # see notebook Part 2
    spark  = build_spark(args.env, cfg['shuffle_parts'])
    t0     = time.time()

    log.info(json.dumps({'event': 'start', 'date': args.date, 'env': args.env}))

    try:
        bronze = spark.read.parquet(f's3a://{args.input_bucket}/bronze/orders/date={args.date}/')
        log.info(json.dumps({'event': 'bronze_read', 'rows': bronze.count()}))

        silver = bronze.pipe(validate).pipe(deduplicate).pipe(enrich)

        run_dq(silver, cfg['max_null_rate'])

        spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')
        silver.withColumn('date', F.lit(args.date)) \\
              .write.mode('overwrite').partitionBy('date') \\
              .parquet(f's3a://{args.output_bucket}/silver/orders/')

        log.info(json.dumps({'event': 'complete', 'rows': silver.count(),
                             'duration_s': round(time.time() - t0, 2)}))
        sys.exit(0)

    except Exception as e:
        log.error(json.dumps({'event': 'failed', 'error': str(e)}))
        sys.exit(1)   # non-zero → EMR step FAILED → Airflow detects

    finally:
        spark.stop()

if __name__ == '__main__':
    main()
══════════════════════════════════════════════════════════════
""")

## Part 5: Production Readiness Checklist

In [ ]:
CHECKLIST = [
    # Correctness
    ("Unit tests for all transformation functions", "testing"),
    ("Integration tests for file read/write",        "testing"),
    ("Test coverage >= 80%",                         "testing"),
    ("DQ checks after each layer",                   "quality"),
    ("Dead-letter table for rejected records",       "quality"),

    # Reliability
    ("Idempotent writes (dynamic overwrite or MERGE)",   "reliability"),
    ("Retry logic in Airflow (retries=3, backoff)",      "reliability"),
    ("Non-zero exit on failure (sys.exit(1))",           "reliability"),
    ("Cluster terminates even on failure (all_done)",    "reliability"),
    ("Checkpoint on durable storage (S3/HDFS)",         "reliability"),

    # Observability
    ("Structured JSON logging (ts, level, pipeline, layer, run_id)", "observability"),
    ("Row counts logged at each layer",                              "observability"),
    ("Spark event logs enabled (History Server)",                    "observability"),
    ("CloudWatch metrics for job duration + DQ",                     "observability"),
    ("Alerts with runbooks for every metric",                        "observability"),
    ("Data freshness SLA check after Gold",                          "observability"),

    # Configuration
    ("No hardcoded paths or environment-specific values",            "config"),
    ("CLI args for date, env, input/output paths",                   "config"),
    ("Secrets in Secrets Manager (not in code or env vars)",        "config"),
    ("Per-env config dict (dev/staging/prod)",                       "config"),

    # CI/CD
    ("CI: lint + test on every PR",                                  "cicd"),
    ("CD: package + upload to S3 on merge to main",                  "cicd"),
    ("OIDC auth to AWS (no static credentials in CI)",               "cicd"),
    ("Version in S3 path (git short hash)",                          "cicd"),

    # Documentation
    ("README: how to run locally, how to deploy",                    "docs"),
    ("Each function has a 1-line docstring explaining WHY",           "docs"),
    ("Architecture diagram or reference document",                    "docs"),
]

from collections import defaultdict
by_category = defaultdict(list)
for item, cat in CHECKLIST:
    by_category[cat].append(item)

print("=== Production Readiness Checklist ===")
for cat, items in by_category.items():
    print(f"\n[{cat.upper()}]")
    for item in items:
        print(f"  ☐ {item}")

print(f"\nTotal: {len(CHECKLIST)} items")

## Exercises

1. Take any Spark script from earlier in this course and make it production-ready: add CLI args, structured logging, config dict per env, DQ check, `sys.exit(1)` on failure. Compare the before and after.
2. Write a `get_secret()` function using `boto3` that fetches from AWS Secrets Manager, handles `ClientError` gracefully (secret not found, permission denied), and logs the secret name (not the value). Write a pytest test for it using `unittest.mock.patch`.
3. Create a `PipelineConfig` dataclass with all environment-specific settings. Write a `load_config(env, date)` function that reads from the config dict and validates required fields. Write tests for: valid env, invalid env, missing date.
4. Walk through the production readiness checklist for a pipeline you've built in this course. How many items are checked? Which category has the most gaps?
5. You join a new team and their Spark pipeline has no tests, hardcoded paths, no logging, and retries 0 times. Prioritize the top 5 improvements to make in the first sprint. Justify the order based on risk.